In [1]:
!pip install albumentations tqdm

   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   -------------- ------------------------- 14.4/38.9 MB 69.7 MB/s eta 0:00:01
   ---------------------------------- ----- 33.3/38.9 MB 81.2 MB/s eta 0:00:01
   ---------------------------------------- 38.9/38.9 MB 68.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 105.7 MB/s eta 0:00:00

   ----------------- ---------------------- 4/9 [opencv-python-headless]
   ----------------- ---------------------- 4/9 [opencv-python-headless]
   ----------------- ---------------------- 4/9 [opencv-python-headless]
   -------------------------- ------------- 6/9 [pydantic]
   ------------------------------- -------- 7/9 [albucore]
   ----------------------------------- ---- 8/9 [albumentations]
   ---------------------------------------- 9/9 [albumentations]



In [12]:
import os 
import glob
import shutil
import xml.etree.ElementTree as ET
import cv2
import random
import albumentations as A
from tqdm import tqdm
from collections import defaultdict

In [13]:
input_base = r"C:\ai_project01\path_guide_dataset\dataset01"
output_base = r"C:\ai_project01\path_guide_dataset\dataset02"
output_labels=os.path.join(output_base,"labels")
output_images=os.path.join(output_base,"images")

In [14]:
class_list = [
    'person',            # 사람입니다.
    'bicycle',           # 자전거입니다.
    'car',               # 자동차입니다.
    'motorcycle',        # 오토바이입니다.
    'bus',               # 버스입니다.
    'truck',             # 트럭입니다.
    'movable_signage',   # 이동식 표지판입니다. (예: 공사 중 표지판)
    'bollard',           # 볼라드입니다. (차량 진입을 막는 기둥)
    'fire_hydrant',      # 소화전입니다.
    'bench',             # 벤치입니다.
    'chair',             # 의자입니다.
    'table',             # 테이블입니다.
    'tree_trunk',        # 나무 기둥입니다.
    'potted_plant',      # 화분입니다.
    'traffic_light',     # 신호등입니다.
    'traffic_sign'       # 도로 표지입니다.
]

In [18]:
class_counts=defaultdict(int)
os.makedirs(output_labels,exist_ok=True)
os.makedirs(output_images,exist_ok=True)

xml_files = glob.glob(os.path.join(input_base,"**","*.xml"),recursive=True)
xml_files

['C:\\ai_project01\\path_guide_dataset\\dataset01\\Bbox_2201\\P1025_07.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2201\\P1025_07.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2202\\P1025_08.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2203\\P1025_09.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2204\\P1025_10.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2205\\P1025_11.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2206\\P1025_12.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2207\\P1025_13.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_new\\Bbox_2208\\P1025_14.xml',
 'C:\\ai_project01\\path_guide_dataset\\dataset01\\인도보행 영상\\바운딩박스\\Bbox_27_

In [19]:
for xml_file in xml_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for image in root.findall("image"):
        image_name = image.attrib["name"]
        img_w = float(image.attrib["width"])
        img_h = float(image.attrib["height"])
        label_path = os.path.join(output_labels, os.path.splitext(image_name)[0] + ".txt")
        with open(label_path, "w") as f:
            for box in image.findall("box"):
                label = box.attrib["label"]
                if label not in class_list:
                    continue
                class_id = class_list.index(label)
                xtl = float(box.attrib["xtl"])
                ytl = float(box.attrib["ytl"])
                xbr = float(box.attrib["xbr"])
                ybr = float(box.attrib["ybr"])
                # 유효성 검사
                if xtl >= xbr or ytl >= ybr:
                    continue
                if xtl < 0 or ytl < 0 or xbr > img_w or ybr > img_h:
                    continue
                if (xbr - xtl) < 1 or (ybr - ytl) < 1:
                    continue
                # 클래스 등장 횟수 증가
                class_counts[label] += 1
                # YOLO 형식으로 변환 (비율 기준)
                cx = (xtl + xbr) / 2 / img_w
                cy = (ytl + ybr) / 2 / img_h
                w = (xbr - xtl) / img_w
                h = (ybr - ytl) / img_h
                f.write(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
        # 이미지 파일 복사
        src_img = os.path.join(os.path.dirname(xml_file), image_name)
        dst_img = os.path.join(output_images, image_name)
        if os.path.exists(src_img):
            shutil.copy2(src_img, dst_img)

In [20]:
print(" ***클래스별 객수 통계*** ")
for label in class_list:
    print(f"{label:20s}:{class_counts[label]}")

 ***클래스별 객수 통계*** 
person              :1210
bicycle             :184
car                 :3213
motorcycle          :51
bus                 :363
truck               :1149
movable_signage     :740
bollard             :1673
fire_hydrant        :114
bench               :189
chair               :133
table               :134
tree_trunk          :4297
potted_plant        :51
traffic_light       :1525
traffic_sign        :1284
